## Transformaciones: recuento

Qué quedó en `data/final/` y `data/artefactos/`, con qué columnas, y qué usa cada técnica

- **Corre en:** local (pyarrow). Solo lee los metadatos de los parquet (filas y esquema), sin procesar datos, así que no
  necesita Spark

### Imports

In [1]:
import json
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_colwidth", 120)
proyecto = Path.cwd()
while not (proyecto / "docker-compose.yml").exists() and proyecto != proyecto.parent:
    proyecto = proyecto.parent

---
### 1. Tablas y artefactos

In [2]:
def recorrer(base):
    filas = []
    for exito in sorted(base.rglob("_SUCCESS")):
        d = exito.parent
        archivos = list(d.rglob("*.parquet"))
        esquema = pq.read_schema(archivos[0])
        filas.append((str(d.relative_to(base)), sum(pq.ParquetFile(f).metadata.num_rows for f in archivos),
                      len(esquema.names), round(sum(f.stat().st_size for f in archivos) / 1024**2, 1),
                      ", ".join(esquema.names)))
    return pd.DataFrame(filas, columns=["tabla", "filas", "columnas", "mb", "nombres"])

final = recorrer(proyecto / "data" / "final")
artefactos = recorrer(proyecto / "data" / "artefactos")
final

,tabla,filas,columnas,mb,nombres
0,entidades,2996,4,0.1,"entidad_key, nombre, ruc, region"
1,er_mapa_lsh,2,6,0.0,"nombre_norm, menciones, clave, nombre_b, jaccard, metodo"
2,er_posibles_vinculadas,71,5,0.0,"nombre_b, nombre_a, jaccard, clave_a, clave_b"
3,hechos,202966,17,16.1,"ocid, award_id, fecha, entidad_key, proveedor_key, region, categoria, metodo_detalle, postor_unico, es_directa, n_po..."
4,participantes,2906242,4,25.9,"ocid, proveedor_key, en_lista_oferentes, gano"
5,procesos,235954,18,61.9,"ocid, fecha, entidad_id_raw, entidad_ruc, entidad, region, descripcion, items_desc, metodo_detalle, categoria, monto..."
6,proveedores,260076,5,4.8,"proveedor_key, nombre, tipo, es_consorcio, region"


In [3]:
artefactos

,tabla,filas,columnas,mb,nombres
0,features_entidad,2996,20,0.4,"entidad_key, n_procesos, pct_postor_unico, pct_directa, postores_promedio, pct_sin_adjudicar, pct_bienes, pct_servic..."
1,features_proveedor,86017,24,5.4,"proveedor_key, n_procesos, n_regiones, monto_total, monto_mediano, pct_postor_unico, pct_directa, pct_bienes, pct_se..."
2,grafo_bipartito/aristas,136099,6,2.1,"src, dst, n_adj, monto, anio_min, anio_max"
3,grafo_bipartito/vertices,89009,5,2.2,"id, tipo, nombre, region, es_consorcio"
4,grafo_coparticipacion/aristas,1639945,4,22.8,"src, dst, n_coparticipa, peso"
5,grafo_coparticipacion/vertices,60213,5,1.7,"id, tipo, nombre, region, es_consorcio"
6,indice_entidad,2992,2,0.0,"idx, clave"
7,indice_familia,386,2,0.0,"idx, clave"
8,indice_proveedor,86017,2,0.8,"idx, clave"
9,matriz_entidad_familia,64240,7,0.8,"entidad_key, familia, anio, n_procesos, monto, entidad_idx, familia_idx"


- **`data/final/`:**
  - `hechos`: 202,966 filas al grano entidad–proveedor–familia
  - `participantes`: 2.9M filas
  - `procesos`: 235,954 procesos limpios
  - dimensiones: 2,996 entidades y 260,076 proveedores, que incluye oferentes que nunca ganaron
- **`data/artefactos/`:**
  - features de 86,017 proveedores ganadores y 2,996 entidades
  - matrices en formato largo con `anio` (159,122 y 64,240 filas)
  - grafo bipartito (89,009 nodos)
  - grafo de co-participación (60,213 nodos)
- Todo suma ~150 MB en Parquet, frente a 3.9 GB del JSON original

---
### 2. Columnas por técnica

| Técnica (Parte) | Artefacto | Columnas que usa |
|---|---|---|
| Clustering y anomalías (II) | `features_proveedor` | `features_raw` (17): `log_n_procesos`, `n_entidades`, `n_regiones`, `log_monto_total`, `log_monto_mediano`, `pct_postor_unico`, `pct_directa`, `pct_bienes`, `pct_servicios`, `pct_obras`, `meses_activos`, `share_top_entidades`, `hhi_entidades`, `n_familias`, `compite`, `tasa_exito`, `es_consorcio` |
| | `features_entidad` | `features_raw` (14): `log_n_procesos`, `pct_postor_unico`, `pct_directa`, `postores_promedio`, `pct_sin_adjudicar`, `pct_bienes`, `pct_servicios`, `pct_obras`, `log_monto_total`, `log_monto_mediano`, `share_top_proveedores`, `hhi_proveedores`, `n_proveedores`, `n_familias` |
| Recomendación (III) | `matriz_entidad_proveedor` | `entidad_idx`, `proveedor_idx`, `anio`, `n_adj` o `senal_log_monto` |
| | `matriz_entidad_familia` | `entidad_idx`, `familia_idx`, `anio`, `n_procesos` o `monto` |
| | `procesos` (final) | `descripcion`, `items_desc` (contenido / TF-IDF) |
| PageRank y HITS (IV) | `grafo_bipartito` | vértices `id`, `tipo`; aristas `src`, `dst`, `n_adj`, `monto` |
| Comunidades (V) | `grafo_coparticipacion` | vértices `id`, `region`; aristas `src`, `dst`, `peso`, `n_coparticipa` |
| Representación (VI) | matriz entidad × proveedor + `grafo_coparticipacion` | SVD sobre `n_adj`; Node2Vec sobre `peso` |
| Ética y auditoría (VII) | `er_mapa_lsh`, `er_posibles_vinculadas` | método de enlace, Jaccard, claves vinculadas |

---
### 3. Conteos finales

In [4]:
leer_json = lambda carpeta, nb, t: pd.DataFrame(json.load(open(proyecto / carpeta / "temp" / nb / "web" / f"{t}.json")))
reconciliacion = leer_json("01_Trans", "b_transformaciones", "reconciliacion")
resumen = leer_json("01_Trans", "c_uso", "resumen_artefactos")
resumen["valor"] = resumen.valor.map(lambda v: f"{v:.4%}" if v < 1 else f"{v:,.0f}")
reconciliacion

,paso,filas
0,procesos staging,236229
1,procesos sin ocid duplicado,236229
2,procesos con fecha en 2023-2025,235954
3,procesos limpios,235954
4,adjudicaciones staging (award × proveedor),203486
5,adjudicaciones en procesos limpios,203293
6,adjudicaciones con proveedor resuelto (award × proveedor),199377
7,hechos: award × proveedor distintos,199377
8,hechos: filas (× familia),202966
9,oferentes staging,2906469


In [5]:
resumen

,medida,valor
0,proveedores (features),"86,017"
1,entidades (features),"2,996"
2,pares entidad × proveedor,"136,099"
3,densidad entidad × proveedor,0.0529%
4,pares entidad × familia,"39,348"
5,densidad entidad × familia,3.4070%
6,vértices bipartito,"89,009"
7,aristas bipartito,"136,099"
8,vértices co-participación,"60,213"
9,aristas co-participación,"1,639,945"


- La reconciliación cierra sin pérdidas no explicadas: cada caída corresponde a un filtro documentado (fecha fuera de rango,
  award sin proveedor)
- **Bipartito:** 136,099 aristas, densidad 0.053%
- **Entidad × familia:** 39,348 pares, densidad 3.4%
- **Co-participación:** 1.64M aristas

---
### 4. Limitaciones

- **Consorcios sin miembros:** el consorcio es un nodo propio (`CONS:`); no se puede atribuir su adjudicación a las
  empresas que lo forman
- **Sin estado del award:** no se excluyen adjudicaciones anuladas
- **Sin historia:** solo el `compiledRelease` (estado final); no se ven modificaciones de monto ni re-adjudicaciones
- **Alcance de `tenderers`:** la fuente no distingue si un oferente solo se registró o llegó a presentar oferta. Las
  constructoras más activas aparecen en miles de procesos (hubs de grado > 12,000), lo que puede inflar la co-participación
- **Cola larga:** el 84.9% de proveedores tiene una sola entidad, lo que limita la recomendación y el clustering para la
  mayoría de proveedores
- **Personas naturales (`PN:`):** contienen DNI; solo se publican agregados

---
### 5. Uso de la data

```python
hechos   = spark.read.parquet("data/final/hechos").where("anio = 2025")            # poda por partición
feat     = spark.read.parquet("data/artefactos/features_proveedor")                # columna features_raw
m_ep     = spark.read.parquet("data/artefactos/matriz_entidad_proveedor")          # COO con anio
train    = m_ep.where("anio <= 2024").groupBy("entidad_idx", "proveedor_idx").sum("n_adj")
aristas  = spark.read.parquet("data/artefactos/grafo_bipartito/aristas")           # src/dst para GraphFrames
```